# Experiment 9b - Conditional spike lift, honestly validated

Follow-up to 9a (Frontier 1). 9a found that *global* loss/target changes can't beat
WAPE (they tax the 95% bulk to lift the 5% tail), but a **conditional** lift -
apply the high-quantile prediction only to rows the model already predicts high -
cleared the 0.05 floor in-sample (CV-WAPE 14.676, +0.062).

**The problem with that number:** the threshold (top-10% of predicted rows) was
chosen on the *same* OOF it was scored on. That is in-sample hyperparameter
selection and is optimistic. This notebook re-measures the lift with **nested,
out-of-fold threshold selection** so the reported gain is trustworthy.

This is a pure post-processing analysis on OOF predictions already saved - no
retraining, no GPU. It does **not** itself produce the final model; it decides
whether the spike-lift is worth folding into the deep-learning model (the
quantile-head gated MLP) as the next step.

**Protocol (nested CV for a single hyperparameter):**

For each test fold `k`:
1. On the *other* folds, sweep the lift fraction `f` (incl. `f=0` = no lift =
   anchor) and pick `f*` minimizing their combined WAPE.
2. Compute the lift threshold as the `(1-f*)` quantile of the base prediction on
   those other folds (never using fold `k`).
3. Apply: on fold `k`, replace the base prediction with the high-quantile arm for
   rows whose base prediction `>=` that threshold; keep base elsewhere.

The aggregate WAPE over the held-out folds is the honest estimate. `f=0` in the
candidate set lets the method fall back to the anchor when lifting doesn't help on
the training folds - so it cannot do worse than the anchor by construction (up to
threshold-transfer noise).


---
## §0. Configuration

In [ ]:
from pathlib import Path

# This notebook reads OOF CSVs already committed to the repo (no raw data / GPU needed).
OOF_9A = Path("experiment-9/oof_predictions_9a_spike.csv")   # has pred_mae, pred_pinball65, pred_logpinball65
OOF_7B = Path("experiment-7/oof_predictions_7b_blend.csv")   # has pred_mlp, pred_blend (DL spike-structure check)
OUTPUT_DIR = Path("experiment-9")
# Kaggle: upload the two CSVs and point these at /kaggle/input/...

N_FOLDS      = 3
BASE_ARM     = "pred_mae"                       # the WAPE-optimal median model = base
LIFT_ARMS    = ["pred_pinball65", "pred_logpinball65"]
LIFT_FRACS   = [0.0, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30]   # f=0 -> anchor fallback
SPIKE_PCT    = 0.05                             # top-5% by true target = "spike rows"
ACCEPT_DELTA = 0.05                             # honest CV gain must exceed this to be a real lead

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"base arm   : {BASE_ARM}")
print(f"lift arms  : {LIFT_ARMS}")
print(f"fractions  : {LIFT_FRACS}  (0.0 = no lift / anchor fallback)")
print(f"accept     : honest nested-CV WAPE must beat anchor by > {ACCEPT_DELTA}")


---
## §1. Setup & load OOF

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

def wape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    return np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + eps) * 100

oof = pd.read_csv(OOF_9A, parse_dates=["timestamp"])
folds = sorted(oof["fold"].unique())
assert len(folds) == N_FOLDS
anchor_cv = np.mean([wape(g["target"], g[BASE_ARM]) for _, g in oof.groupby("fold")])
print(f"9a OOF: {oof.shape}, folds={folds}")
print(f"anchor ({BASE_ARM}) CV-WAPE = {anchor_cv:.3f}")
for arm in LIFT_ARMS:
    w = np.mean([wape(g["target"], g[arm]) for _, g in oof.groupby("fold")])
    print(f"  {arm:<18} solo CV-WAPE = {w:.3f}")


---
## §2. Conditional-lift utilities + nested-CV selector

In [ ]:
def conditional_lift(df, base_col, lift_col, threshold):
    "Replace base with lift where base prediction >= threshold; keep base elsewhere."
    base = df[base_col].to_numpy(float)
    lift = df[lift_col].to_numpy(float)
    return np.where(base >= threshold, lift, base)

def cv_wape_of(pred_by_fold):
    "pred_by_fold: dict fold -> (y, p). Returns mean per-fold WAPE."
    return float(np.mean([wape(y, p) for (y, p) in pred_by_fold.values()]))

def insample_best(df, base_col, lift_col, fracs):
    "9a-style: pick the fraction minimizing WAPE on the SAME data (optimistic)."
    best = (0.0, None)
    for f in fracs:
        thr = np.inf if f == 0.0 else np.quantile(df[base_col], 1 - f)
        p = conditional_lift(df, base_col, lift_col, thr)
        pf = [wape(g["target"], conditional_lift(g, base_col, lift_col, thr))
              for _, g in df.groupby("fold")]
        w = float(np.mean(pf))
        if best[1] is None or w < best[1]:
            best = (f, w)
    return best  # (frac, in-sample CV-WAPE)

def nested_cv(df, base_col, lift_col, fracs):
    "Honest: for each test fold, pick frac+threshold on the OTHER folds only."
    held = {}
    chosen = {}
    for k in folds:
        train = df[df["fold"] != k]
        test  = df[df["fold"] == k]
        # select f* on training folds
        best_f, best_w = 0.0, np.inf
        for f in fracs:
            thr = np.inf if f == 0.0 else np.quantile(train[base_col], 1 - f)
            w = wape(train["target"], conditional_lift(train, base_col, lift_col, thr))
            if w < best_w:
                best_w, best_f = w, f
        thr_star = np.inf if best_f == 0.0 else np.quantile(train[base_col], 1 - best_f)
        held[k] = (test["target"].to_numpy(float),
                   conditional_lift(test, base_col, lift_col, thr_star))
        chosen[k] = (best_f, thr_star)
    return cv_wape_of(held), chosen

print("utilities ready.")


---
## §3. Confirmation - in-sample (9a-style) vs honest nested CV

The gap between the two columns is the optimism in 9a's number. The **nested** column is the one that decides whether the lift is a real lead.

In [ ]:
rows = []
for lift_col in LIFT_ARMS:
    f_in, w_in = insample_best(oof, BASE_ARM, lift_col, LIFT_FRACS)
    w_nested, chosen = nested_cv(oof, BASE_ARM, lift_col, LIFT_FRACS)
    rows.append(dict(
        lift=lift_col, insample_frac=f_in, insample_wape=w_in,
        nested_wape=w_nested,
        nested_delta=anchor_cv - w_nested,
        chosen_fracs=tuple(chosen[k][0] for k in folds),
    ))
res = pd.DataFrame(rows)

print(f"anchor CV-WAPE = {anchor_cv:.3f}\n")
print(f"{'lift arm':<18} {'in-sample':>10} {'(frac)':>7} {'NESTED':>8} {'Δ honest':>9}  chosen f per fold")
print("-" * 78)
for _, r in res.iterrows():
    flag = "  REAL LEAD" if r['nested_delta'] > ACCEPT_DELTA else \
           ("  ~noise" if r['nested_delta'] > -1e-9 else "  worse")
    print(f"{r['lift']:<18} {r['insample_wape']:>10.3f} {r['insample_frac']:>7.2f} "
          f"{r['nested_wape']:>8.3f} {r['nested_delta']:>+9.3f}  {r['chosen_fracs']}{flag}")

print("\nReading this:")
print("  - in-sample == 9a's optimistic number; NESTED == the honest one.")
print("  - if 'chosen f per fold' is stable (same f on all 3 folds), the threshold transfers;")
print("    if it jumps around, the lift fraction is unstable and the lead is fragile.")


---
## §4. Does the same spike under-prediction exist on the DL model?

9a's evidence used the LGBM blend. Confirm the *deep-learning* model (the MLP OOF from 7b) under-predicts the same spike rows - this is what justifies building the lift into a DL quantile head rather than only post-processing the tree.

In [ ]:
dl = pd.read_csv(OOF_7B, parse_dates=["timestamp"])
dl_cols = [c for c in ["pred_mlp", "pred_blend"] if c in dl.columns]
thr_t = dl["target"].quantile(1 - SPIKE_PCT)
spike = dl["target"] >= thr_t
print(f"DL OOF: {dl.shape}; spike rows (top {SPIKE_PCT*100:.0f}% target, >= {thr_t:.2f}): {int(spike.sum())}\n")
print(f"{'model':<12} {'overall resid':>14} {'spike resid':>13} {'spike-WAPE':>11}")
print("-" * 54)
for c in dl_cols:
    res_all   = (dl['target'] - dl[c]).mean()
    res_spike = (dl.loc[spike, 'target'] - dl.loc[spike, c]).mean()
    sw = wape(dl.loc[spike, 'target'], dl.loc[spike, c])
    print(f"{c:<12} {res_all:>+14.3f} {res_spike:>+13.3f} {sw:>11.3f}")
print("\n(positive resid = under-prediction. If spike resid is large & positive on the MLP")
print(" too, the DL model has the same tail gap -> a quantile head is the principled fix.)")


---
## §5. Save + verdict

In [ ]:
res_path = OUTPUT_DIR / "conditional_lift_9b.csv"
res.to_csv(res_path, index=False)
print(f"results -> {res_path}")

best = res.loc[res["nested_wape"].idxmin()]
print("\n=== Verdict (Frontier 1, conditional lift) ===")
print(f"  anchor CV-WAPE          : {anchor_cv:.3f}")
print(f"  best honest nested-CV   : {best['nested_wape']:.3f} via {best['lift']} "
      f"(Δ {best['nested_delta']:+.3f}, chosen f {best['chosen_fracs']})")

if best["nested_delta"] > ACCEPT_DELTA:
    print("\n  REAL LEAD: the conditional spike-lift survives honest out-of-fold selection.")
    print("  -> Build the DL form: gated MLP (7e harness) with a QUANTILE / asymmetric-loss head.")
    print("     The network learns the per-row conditional quantile, doing the lift natively")
    print("     (no hand-set threshold). Use q0.5 head for the WAPE point forecast and a")
    print("     higher-quantile head, gated on predicted level, for the spike rows.")
    print("     That is the deep-learning deliverable; this notebook proved the lift is worth it.")
else:
    print("\n  NOT CONFIRMED: 9a's +0.062 was largely in-sample optimism.")
    print("  -> Don't ship a post-hoc lift. Two honest options:")
    print("     (a) still build the DL quantile-head MLP - a learned per-row quantile may")
    print("         beat a hand-thresholded blend even where the blend failed; OR")
    print("     (b) move to Frontier 2 (Chronos stacking) for genuinely decorrelated signal.")
